In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit  # sigmoid for smooth saturation
import effector
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from interpret import set_visualize_provider, show
from interpret.glassbox import ExplainableBoostingRegressor
from interpret.provider import InlineProvider
from effector.calm.calm import CALMRegressor, RegionalPDPDetector
from sklearn.base import clone
import os
from mpl_toolkits.axes_grid1 import make_axes_locatable
from effector.calm.datasets.bike_sharing import BikeSharing
from effector.calm.blackbox import XGBRegressor
from experiments import set_random_seeds
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
import re
import matplotlib.ticker as ticker
from utils.plot_utils import *
from utils.get_plot_values import (
    calm_shape_value,
    calm_list_conditions,
    calm_shift_1d,
    ga2m_shape_value,
    ga2m_interaction_value,
    ga2m_shift_1d,
    ga2m_shift_interaction_wrt_feat,
    get_min_hr_shift_for_y_shift,
    plot_hr_shift_y_shift
)

# User Study - Synthetic

In [ ]:
# f1(x1 | x3)
def f11(x1):
    return 1.2 * x1**3 + 0.1 * x1  # steeper growth early

def f12(x1):
    return 0.9 * np.tanh(2 * x1)  # saturates later

def f1(x1, x3):
    y = np.zeros_like(x1)

    cond = x3 > 0
    y[cond] = f11(x1[cond])
    y[~cond] = f11(x1[~cond])
    return y

def f21(x2):
    return 0.4 * np.sin(np.pi * x2 / 2 - 0.2) - 0.4  # same shape, lower offset

def f22(x2):
    return 0.6 * np.sin(np.pi * x2 / 2 - 0.1) - 0.2  # same shape, lower offset

def f23(x2):
    return 0.8 * np.sin(np.pi * x2 / 2)  # natural bell-like response

def f2(x2, x1):
    y = np.zeros_like(x2)

    cond1 = x1 < - 0.4
    cond2 = np.logical_and(x1 >= -0.4, x1 < 0.4 )
    cond3 = x1 >= 0.4
    y[cond1] = f21(x2[cond1])
    y[cond2] = f22(x2[cond2])
    y[cond3] = f23(x2[cond3])
    return y


def f3 (x3):
    return  1.5 * (expit(2 * x3) - 0.5)  # maps x3 in [-1,1] to roughly [-0.75, 0.75]


x = np.linspace(-1, 1, 200)


# --- Figure 1: f1(x1 | x3) ---
plt.figure()
plt.plot(x, f11(x), label=r"$f(x_1 \mid x_3 > 0)$")
plt.plot(x, f12(x), label=r"$f(x_1 \mid x_3 \leq 0)$")

plt.axvline(x=-0.4, color='gray', linestyle=':')
plt.axvline(x=0.5, color='gray', linestyle=':')
plt.text(-0.4, -0.9, r'$x_2\ (\uparrow)$', ha='center', va='top', fontsize=12)
plt.text(0.5, -0.9, r'$x_2\ (\downarrow)$', ha='center', va='top', fontsize=12)

plt.xlabel(r"$x_1$")
plt.ylabel(r"$y$")
plt.xticks([-1, -.5, 0, 0.5, 1])
plt.yticks([-1.5, -.75, 0, .75, 1.5])
plt.ylim(-1.5, 1.5)
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 2: f2(x2 | x1) ---
plt.figure()
plt.plot(x, f21(x), label=r"$f(x_2 \mid x_1 < -0.4)$")
plt.plot(x, f22(x), label=r"$f(x_2 \mid x_1 \in [-0.4, 0.4))$")
plt.plot(x, f23(x), label=r"$f(x_2 \mid x_1 \geq 0.4)$")

plt.xlabel(r"$x_2$")
plt.xticks([-1, -.5, 0, 0.5, 1])
plt.yticks([-1.5, -.75, 0, .75, 1.5])
plt.ylabel(r"$y$")
plt.ylim(-1.5, 1.5)
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 3: f3(x3) ---
plt.figure()
plt.plot(x, f3(x), label=r"$f_3(x_3)$")
plt.axvline(x=0, color='gray', linestyle=':')
plt.text(0, -0.9, r'$x_1\ (\updownarrow)$', ha='center', va='top', fontsize=12)

plt.xlabel(r"$x_3$")
plt.ylabel(r"$y$")
plt.xticks([-1, -.5, 0, 0.5, 1])
plt.yticks([-1.5, -.75, 0, .75, 1.5])
plt.ylim(-1.5, 1.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit
import os

def f11(x1): return 1.2 * x1**3 + 0.1 * x1
def f12(x1): return 0.9 * np.tanh(2 * x1)
def f1(x1, x3):
    y = np.zeros_like(x1)
    cond = x3 > 0
    y[cond] = f11(x1[cond])
    y[~cond] = f12(x1[~cond])
    return y

def f21(x2): return 0.4 * np.sin(np.pi * x2 / 2 - 0.2) - 0.4
def f22(x2): return 0.6 * np.sin(np.pi * x2 / 2 - 0.1) - 0.2
def f23(x2): return 0.8 * np.sin(np.pi * x2 / 2)
def f2(x2, x1):
    y = np.zeros_like(x2)
    cond1 = x1 < -0.4
    cond2 = np.logical_and(x1 >= -0.4, x1 < 0.4)
    cond3 = x1 >= 0.4
    y[cond1] = f21(x2[cond1])
    y[cond2] = f22(x2[cond2])
    y[cond3] = f23(x2[cond3])
    return y

def f3(x3): return 1.5 * (expit(2 * x3) - 0.5)

x = np.linspace(-1, 1, 200)

fontsize = 13
labelsize = 13
handletextpad=0.2
# Subplots
fig, axs = plt.subplots(1, 3, figsize=(10, 3.8))
# --- Subplot 1: f1(x1 | x3) ---
axs[0].plot(x, f11(x), label=r"$f(x_1 \mid x_3 > 0)$")
axs[0].plot(x, f12(x), label=r"$f(x_1 \mid x_3 \leq 0)$")
axs[0].axvline(x=-0.4, color='gray', linestyle=':')
axs[0].axvline(x=0.4, color='gray', linestyle=':')
axs[0].text(-0.4, -0.9, r'$x_2\ (\uparrow)$', ha='center', va='top', fontsize=fontsize+1)
axs[0].text(0.4, -0.9, r'$x_2\ (\uparrow)$', ha='center', va='top', fontsize=fontsize+1)
axs[0].set_xlabel(r"$x_1$", fontsize=labelsize+1)
axs[0].set_ylabel(r"$y$", fontsize=labelsize)
axs[0].set_xticks([-1, -.5, 0, 0.5, 1])
axs[0].set_yticks([-1.5, -.75, 0, .75, 1.5])
axs[0].set_ylim(-1.5, 1.5)
axs[0].legend(
    fontsize=fontsize,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),  
    handlelength=1.5,
    handletextpad=handletextpad,
)

# --- Subplot 2: f2(x2 | x1) ---
axs[1].plot(x, f21(x), label=r"$f(x_2 \mid x_1 < -0.4)$")
axs[1].plot(x, f22(x), label=r"$f(x_2 \mid x_1 \in [-0.4, 0.4))$")
axs[1].plot(x, f23(x), label=r"$f(x_2 \mid x_1 \geq 0.4)$")
axs[1].set_xlabel(r"$x_2$", fontsize=labelsize+1)
axs[1].set_xticks([-1, -.5, 0, 0.5, 1])
axs[1].set_yticks([-1.5, -.75, 0, .75, 1.5])
axs[1].set_ylabel(r"$y$", fontsize=labelsize)
axs[1].set_ylim(-1.5, 1.5)
axs[1].legend(fontsize=fontsize, loc='best', handlelength=1, handletextpad=handletextpad, framealpha=0.4 ,labelspacing=0.08)
axs[1].legend(
    fontsize=fontsize,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),  
    handlelength=1.5,
    handletextpad=handletextpad,
)

# --- Subplot 3: f3(x3) ---
axs[2].plot(x, f3(x), label=r"$f_3(x_3)$")
axs[2].axvline(x=0, color='gray', linestyle=':')
axs[2].text(0, -0.9, r'$x_1\ (\updownarrow)$', ha='center', va='top', fontsize=fontsize+1)
axs[2].set_xlabel(r"$x_3$", fontsize=labelsize+1)
axs[2].set_ylabel(r"$y$", fontsize=labelsize)
axs[2].set_xticks([-1, -.5, 0, 0.5, 1])
axs[2].set_yticks([-1.5, -.75, 0, .75, 1.5])
axs[2].set_ylim(-1.5, 1.5)
axs[2].legend(
    fontsize=fontsize,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),  
    handlelength=1.5,
    handletextpad=handletextpad,
)


for ax in axs:
    ax.tick_params(axis='both', labelsize=labelsize-1)

plt.tight_layout()
plt.show()


### Plot for x₁ - Interpretation

The curves illustrate the different ways in which a feature can affect the output y. In particular, feature x₁ influences the output y in two distinct ways, depending on the value of x₃:
* when x₃ > 0, the blue curve applies.
* when x₃ ≤ 0, the orange curve applies.

These curves show the contribution of x₁ to the output in each case. For example:
* If x₃ > 0, (blue curve), for x₁ = -0.5, the contribution is approximately -0.2.
* If x₃ ≤ 0 (orange curve), for x₁ = -0.5, the contribution of x₁ to y is approximately -0.75.

The curves also show how changes in x₁ affect the output y. For example:

* For x₃ > 0 (blue curve), a change from x₁ = 0 to x₁ = 0.3 changes the output by (approximately) Dy = 0 - 0 = 0
* Fox x₃ ≤ 0 (orange curve),  a change from x₁ = 0 to x₁ = 0.3 changes the output by (approximately) Dy = 0.5 - 0 = 0.5 

Pay special attention to the vertical dotted lines (at x₁ = -0.4 and x₁ = 0.5). These represent discontinuities in the effect of x₁. For example, moving from x₁ = -0.5 to x₁ = -0.3, the orange curve illustrates a change from -0.8 to -0.6, suggesting Dy = +0.2. However, because this interval crosses a dotted line, there is also a jump in the output. The actual change is:

* Larger than illustrated (Dy > 0.2) if there's an upward arrow near the line
* Smaller than illustrated (Dy < 0.2) if there's a downward arrow on the line
* Uncertain (can be either greater than or less than 0.2) if there's an up-down arrow

Therefore, when analyzing how a change in x₁ affects the output y, always check whether the change crosses a vertical dotted line:

* If not, the change in output can be read directly from the curve.
* If it does, the change is different than the one illustrated. Refer to the arrow to understand the true direction and magnitude of the change

### Plot for x₂ and x₃ - Interpretation

In a similar way, you can interpret the contribution and effect of x₂ and x₃ from their respective plots.

* x₂ affects the output in three different ways, depending on the value of x₁. However, there are no discontinuities in its effect — that is, no vertical dotted lines appear in the plot.

* x₃ influences the output in a single way, but there is one discontinuity at  x₃  = 0, with the direction of the jump unknown (as indicated by the up-down arrow).

# User Study - Bike Sharing

For the real dataset case we analyze predictions from a model trained on a Bike Sharing dataset. The model predicts the number of rented bikes based on various features, including the hour of the day.

The plot shows the contribution of the feature hour to the prediction.
The shape of this contribution changes depending on four conditions (regions), based on whether it is a working day, the temperature, and the year.

Each curve in the plot corresponds to a different region.
Dashed vertical lines mark region-specific change points, and the x-axis represents the hour of the day

## Load - Preprocess Dataset

In [ ]:
random_seed = 42
set_random_seeds(random_seed)

dataset = BikeSharing()
dataset.fetch()
dataset.preprocess()
X, y = dataset.get_Xy()

features, numerical_features, categorical_features = dataset.get_feature_names()
numerical_features.append("hr")
categorical_features.remove("hr")
orig_X = X.copy()
orig_X_df = pd.DataFrame(orig_X, columns=features)

t = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(), categorical_features),
        ("num", StandardScaler(), numerical_features),
    ]
)

X = t.fit_transform(X)

features = categorical_features + numerical_features

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

label_maps = {
    "season": {1: "winter", 2: "spring", 3: "summer", 4: "fall"},
    "yr": {0: "2011", 1: "2012"},
    "mnth": {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
             7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"},
    "holiday": {0: "No", 1: "Yes"},
    "weekday": {0: "Sunday", 1: "Monday", 2: "Tuesday", 3: "Wednesday",
                4: "Thursday", 5: "Friday", 6: "Saturday"},
    "workingday": {0: "No", 1: "Yes"},
    "weathersit": {
        1: "Clear/Few clouds", 
        2: "Mist + Cloudy", 
        3: "Light Snow/Rain", 
        4: "Heavy Rain/Snow"
    }
}

feature_types = [
    "ordinal",  # season
    "nominal",  # yr
    "ordinal",  # mnth
    "nominal",  # holiday
    "nominal",  # weekday
    "ordinal",  # workingday
    "nominal",  # weathersit
    "continuous",  # temp
    "continuous",  # hum
    "continuous",  # windspeed
    "continuous"   # hr
]

display_labels_map = {
    "season": "Season",
    "yr": "Year",
    "mnth": "Month",
    "holiday": "Holiday",
    "weekday": "Weekday",
    "workingday": "Working Day",
    "weathersit": "Weather Situation",
    "temp": "Temperature",
    "hum": "Humidity",
    "windspeed": "Wind Speed",
    "hr": "Hour"
}

y_display_label = "Bike Rentals"

##  Apply CALM

In [ ]:
region_detector = RegionalPDPDetector(
    heter_pcg_threshold=0.2,
    nof_splits_numerical=10,
)
calm = CALMRegressor(
    blackbox_model = XGBRegressor,
    masked_gam_name="NoInteractionsEBMRegressor",
    region_detector=region_detector,
    feat_types=feature_types,
)
set_random_seeds(random_seed)

calm.fit(X_train, y_train, feat_labels=features)
score = calm.score(X_test, y_test)
print(score)

for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_level_stats()
    print()

##  Apply GA2M

In [ ]:
ga2m = ExplainableBoostingRegressor(random_state=42, feature_names=features, feature_types=feature_types, interactions=2)
ga2m.fit(X_train, y_train)

## Plot CALM & GA2M Shape Functions for Feature Hour

In [ ]:
feat_idx = features.index("hr")
plot_calm_gam_shape_function(
    gam=None,
    calm=calm,
    ga2m=ga2m,
    feat_idx=feat_idx,
    save_dir=None, #f"bike_sharing_shapes_hr_cont/{features[feat_idx]}",
    figsize=(10, 6),
    display_title=False,
    feat_labels=features,
    ga2m_fixed_figsize=False,
    show_confidence=False,
    t=t,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    foi_type=feature_types[10],
    label_maps=label_maps, 
    display_labels_map=display_labels_map,
    y_display_label=y_display_label

)

### Shape Functions of Features with Condition on Hour

In [ ]:
for feat_idx in [features.index("yr"), features.index("temp"), features.index("hum")]:
    plot_calm_gam_shape_function(
        gam=None,
        calm=calm,
        ga2m=None,
        feat_idx=feat_idx,
        save_dir=None, #f"bike_sharing_shapes_hr_cont/{features[feat_idx]}",
        figsize=(10, 6),
        display_title=True,
        feat_labels=features,
        ga2m_fixed_figsize=False,
        show_confidence=False,
        t=t,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        foi_type=feature_types[feat_idx],
        label_maps=label_maps, 

    )

## Custom Plot For User Study - Hour Feature

The customization of this plot compared to the above original includes reseting labels to be interpretable and add dashed lines where hour is chosen as conditional dividing another feature into regions plus the arrows ($\updownarrow$) to show that we cannot ensure the direction of the change when crossing the borders of the regions

In [ ]:
def plot_custom_shape_functions_multiple(
    models,
    model_features,
    feature_label,
    model_labels=None,
    colors=None,
    title=None,
    save_path=None,
    figsize=(8, 6),
    ylim=None,
    show_confidence=False,
    t=None,
    categorical_features=None,
    numerical_features=None,
    foi_type=None,
    label_maps=None,
    axhlines=None,
    format='pdf',
    fontsize=12,
):

    fig, ax = plt.subplots(figsize=figsize)

    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.grid(True, axis="y", linestyle="-", linewidth=0.5, alpha=0.4)

    if colors is None:
        colors = plt.cm.tab10.colors

    is_categorical = (
        categorical_features is not None and feature_label in categorical_features
    )

    if is_categorical:
        all_categories = set()
        y_by_model = []

        for i, model in enumerate(models):
            x, y, _, _, _, _ = extract_full_shape_function(model, model_features[i])

            if t is not None:
                x = reverse_column_transform(
                    values=x,
                    pipeline=t,
                    feature_name=feature_label,
                    categorical_features=categorical_features,
                    numerical_features=numerical_features,
                )
            if foi_type is not None and foi_type == "ordinal":
                x = x.astype(float).astype(int)

            all_categories.update(x)
            y_by_model.append((x, y))

        all_categories = sorted(all_categories)
        category_to_idx = {cat: idx for idx, cat in enumerate(all_categories)}
        n_categories = len(all_categories)

        bar_centers = np.arange(n_categories)
        n_models = len(models)
        bar_width = 0.8 / n_models

        for i, (cats, y) in enumerate(y_by_model):
            aligned_y = np.zeros(n_categories)
            aligned_y[:] = np.nan

            for cat, val in zip(cats, y):
                idx = category_to_idx[cat]
                aligned_y[idx] = val

            offset = (i - n_models / 2) * bar_width + bar_width / 2
            positions = bar_centers + offset
            color = colors[i % len(colors)]
            label = model_labels[i] if model_labels is not None else None

            ax.bar(
                positions,
                aligned_y,
                width=bar_width,
                label=label,
                color=color,
                alpha=0.8,
            )

        ax.set_xticks(bar_centers)
        if label_maps and feature_label in label_maps:
            all_categories = [
                label_maps[feature_label].get(val, val) for val in all_categories
            ]

        rotation = 45 if len(all_categories) > 10 else 0
        ax.set_xticklabels(all_categories, rotation=rotation, fontsize=fontsize)

    else:
        for i, model in enumerate(models):
            x, y, lower, upper, _, _ = extract_full_shape_function(
                model, model_features[i]
            )
            y = y - np.mean(y)
            x = x.astype(float)

            if t is not None:
                x = reverse_column_transform(
                    values=x,
                    pipeline=t,
                    feature_name=feature_label,
                    categorical_features=categorical_features,
                    numerical_features=numerical_features,
                )
                if feature_label == "temp":
                    x = rev_celsius(x)

            color = colors[i % len(colors)]
            label = model_labels[i] if model_labels is not None else f"Model {i+1}"

            ax.plot(x, y, label=label, color=color, linewidth=2)
            if show_confidence and lower is not None and upper is not None:
                ax.fill_between(x, lower, upper, color=color, alpha=0.2)

    ax.set_xlabel('Hour', fontsize=fontsize+1)
    ax.set_ylabel("Bike Rentals", fontsize=fontsize+1)
    ax.tick_params(
        axis="both",
        labelsize=fontsize,
    )
    ax.text(7.5, -140, r'$(\updownarrow)$', ha='center', va='top', fontsize=fontsize)
    ax.text(9.5, -140, r'$(\updownarrow)$', ha='center', va='top', fontsize=fontsize)

    if axhlines is not None:
        for axhline in axhlines:
            ax.axvline(x=axhline, color='gray', linestyle=':')

        existing_ticks = ax.get_xticks().tolist()
        all_ticks = sorted(set(existing_ticks + axhlines))
        ax.set_xticks(all_ticks[1:])

    if ylim is not None:
        ax.set_ylim(ylim)

    if title:
        plt.title(title)

    ax.legend(fontsize=fontsize)
    fig.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
            format=format,
        )
        print(f"Saved combined shape function plot to {save_path}")
    else:
        plt.show()


def plot_calm_shape_function(
    calm,
    feat_idx,
    figsize=(10, 6),
    show_confidence=False,
    save_dir=None,
    feat_labels=None,
    display_title=True,
    t=None,
    categorical_features=None,
    numerical_features=None,
    foi_type=None,
    label_maps=None,
    axhlines=None,
    format='pdf'
):

    calm_gam, calm_feature_names = calm.masked_gam.model, calm.new_names
    calm_feature_names = simplify_expressions(calm_feature_names)
    if t is not None:
        calm_feature_names = scale_back_expressions(
            calm_feature_names,
            t,
            categorical_features,
            numerical_features,
            labels_map=label_maps,
        )

    calm_features_conditions_names_map = build_feature_mapping(
        calm_feature_names, feat_labels
    )
    calm_feature_names = beautify_condition_latex(calm_feature_names)

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    colors = ["blue", "orange", "purple", "cyan"]

    calm_feature_names = fix_latex_and_operator(calm_feature_names)
    conditions = calm_features_conditions_names_map[feat_idx]
    calm_colors = colors[: len(conditions)]

    all_models = [calm_gam] * len(conditions)
    all_feats = list(conditions)
    ylim = get_global_ylim(all_models, all_feats)

    feature_label = feat_labels[feat_idx]
    model_labels = [
        rf'Hour | Non-WorkingDay & Temperature $\leq$ 11',
        rf'Hour | Non-WorkingDay & Temperature > 11',
        rf'Hour | WorkingDay & Year = 2011',
        rf'Hour | WorkingDay & Year = 2012',
    ]

    plot_custom_shape_functions_multiple(
        models=[calm_gam] * len(conditions),
        model_features=conditions,
        feature_label=feature_label,
        model_labels=model_labels,
        title=f"CALM Shape functions" if display_title else None,
        show_confidence=show_confidence,
        figsize=figsize,
        colors=calm_colors,
        ylim=ylim,
        save_path=(
            os.path.join(save_dir, f"calm_{feature_label}.{format}") if save_dir else None
        ),
        t=t,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        foi_type=foi_type,
        label_maps=label_maps,
        axhlines=axhlines,
        format=format,
    )

feat_idx=10
plot_calm_shape_function(
    calm=calm,
    feat_idx=feat_idx,
    figsize=(10, 6),
    show_confidence=False,
    display_title=False,
    feat_labels=features,
    t=t,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    foi_type=feature_types[feat_idx],
    label_maps=label_maps,
    axhlines=[7,9],
    format='jpg',
    # save_dir=f"bike_sharing_shapes_hr_cont/{features[feat_idx]}_custom",
)

## Get CALM Exact Plot Values for User Study

In [ ]:
# get the conditions and respective IDs for feature "hr"
opts = calm_list_conditions(
    calm, feat_idx=features.index("hr"),
    feat_labels=features,
    t=t,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    label_maps=label_maps,
)

for o in opts:
    print(f"[condition_id={o['pos']}]  |  {o['condition_only']}")


In [ ]:
condition_id = 1  # non-working day & temperature > 11 C

### CALM value at hour = 10:00

In [ ]:
y = calm_shape_value(
    calm, feat_idx=features.index("hr"), feature_value=10,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour=10:00, non-working day with temperature > 11°C value: {int(np.round(y,0))}")

### CALM value at hour = 12:30

In [ ]:
y = calm_shape_value(
    calm, feat_idx=features.index("hr"), feature_value=12.5,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour=12:30, non-working day with temperature > 11°C value: {int(np.round(y,0))}")

### Q15.a. Suppose it is a non-working day with temperature > 11°C and Year = 2012. What is the approximate change in the prediction when hour increases from 10:00 to 12:30?

In [ ]:
calm_shift = calm_shift_1d(
    calm, feat_idx=features.index("hr"), k1=10, k2=12.5,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour shift 10:00->12:30, non-working day with temperature > 11°C delta: {int(np.round(calm_shift,0))}")

### CALM value at hour = 17:00

In [ ]:
y = calm_shape_value(
    calm, feat_idx=features.index("hr"), feature_value=17,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour=17:00, non-working day with temperature > 11°C value: {int(np.round(y,0))}")

### CALM value at hour = 20:00

In [ ]:
y = calm_shape_value(
    calm, feat_idx=features.index("hr"), feature_value=20,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour=20:00, non-working day with temperature > 11°C value: {int(np.round(y,0))}")

### Q16.a. Suppose it is a non-working day with temperature > 11°C and Year = 2012. What is the approximate change in the prediction when hour increases from 17:00 to 20:00?

In [ ]:
calm_shift = calm_shift_1d(
    calm, feat_idx=features.index("hr"), k1=17, k2=20,
    condition_id=condition_id,  # choose the 1st conditional curve
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)
print(f"calm hour shift 17:00->20:00, non-working day with temperature > 11°C delta: {int(np.round(calm_shift,0))}")

### Q17.a. Suppose it is a non-working day with temperature > 11°C and Year = 2012. The current hour is 15:00. What is the shortest amount of time it takes for the predicted rentals to drop by 50?

In [ ]:
min_hour_value = 15
max_hour_value = 22.5
step = 0.1
calm_results = []

for end_hour in np.arange(min_hour_value + step, max_hour_value, step):
        
    calm_shift = calm_shift_1d(
        calm, feat_idx=features.index("hr"), k1=min_hour_value, k2=end_hour,
        condition_id=condition_id,  # choose the 1st conditional curve non-working day & temp > 11C
        feat_labels=features,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        t=t,
    )
    calm_results.append({
        "end_hour": end_hour,
        "out_shift": calm_shift,
        "hr_shift": end_hour - min_hour_value,
    })

In [ ]:
best_result = get_min_hr_shift_for_y_shift(calm_results, y_shift=-50)
print(f"total delta: {int(np.round(best_result['out_shift'],0))}, total hr shift: {best_result['hr_shift']*60:.2f} mins")

In [ ]:
plot_hr_shift_y_shift(calm_results)

## Get GA2M Exact Plot Values for User Study

### GA2M 1D & Heatmap values at hour = 10:00

In [ ]:
hr_feat_idx = features.index("hr")
hr_value = 10
year_value = 1  # 2012
workingday_value = 0  # non-working day

hr_10 = ga2m_shape_value(
    ga2m, feat_idx=hr_feat_idx, feature_value=hr_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

hr_10_year_12 = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    feature_value=hr_value,
    other_value=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

hr_10_day_no_work = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    feature_value=hr_value,
    other_value=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour=10am 2D value: {int(np.round(hr_10,0))}")
print(f"ga2m hour=10am, year=2012 heatmap value: {int(np.round(hr_10_year_12,0))}")
print(f"ga2m hour=10am, day=non-working heatmap value: {int(np.round(hr_10_day_no_work,0))}")

### GA2M 1D & Heatmap values at hour = 12:30 

In [ ]:
hr_feat_idx = features.index("hr")
hr_value = 12.5
year_value = 1  # 2012
workingday_value = 0  # non-working day

hr_12_5 = ga2m_shape_value(
    ga2m, feat_idx=hr_feat_idx, feature_value=hr_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

hr_12_5_year_12 = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    feature_value=hr_value,
    other_value=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

hr_12_5_day_no_work = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    feature_value=hr_value,
    other_value=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour=12.5am 2D value: {int(np.round(hr_12_5,0))}")
print(f"ga2m hour=12.5am, year=2012 heatmap value: {int(np.round(hr_12_5_year_12,0))}")
print(f"ga2m hour=12.5am, day=non-working heatmap value: {int(np.round(hr_12_5_day_no_work,0))}")

### Q15.b. Suppose it is a non-working day with temperature > 11°C and Year = 2012. What is the approximate change in the prediction when hour increases from 10:00 to 12:30?

In [ ]:
min_hour_value = 10.0
max_hour_value = 12.5

delta_hr_10_12_5 = ga2m_shift_1d(
    ga2m, feat_idx=hr_feat_idx, k1=min_hour_value, k2=max_hour_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

delta_hr_10_12_5_year_12 = ga2m_shift_interaction_wrt_feat(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    k1=min_hour_value,
    k2=max_hour_value,
    other_value_fixed=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

delta_hr_10_12_5_day_no_work = ga2m_shift_interaction_wrt_feat(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    k1=min_hour_value,
    k2=max_hour_value,
    other_value_fixed=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour shift 10am->12:30pm 1D delta: {int(np.round(delta_hr_10_12_5,0))}")
print(f"ga2m hour shift 10am->12:30pm, year=2012 heatmap delta: {int(np.round(delta_hr_10_12_5_year_12,0))}")
print(f"ga2m hour shift 10am->12:30pm, day=non-working heatmap delta: {int(np.round(delta_hr_10_12_5_day_no_work,0))}")

print(f"Total shift 10am->12:30pm, year=2012, non-working day: {int(np.round(delta_hr_10_12_5 + delta_hr_10_12_5_year_12 + delta_hr_10_12_5_day_no_work,0))}")

### GA2M 1D & Heatmap values at hour = 17:00 

In [ ]:
hr_feat_idx = features.index("hr")
hr_value = 17.00
year_value = 1  # 2012
workingday_value = 0  # non-working day

hr_17 = ga2m_shape_value(
    ga2m, feat_idx=hr_feat_idx, feature_value=hr_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

hr_17_year_12 = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    feature_value=hr_value,
    other_value=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

hr_17_day_no_work = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    feature_value=hr_value,
    other_value=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour=17pm 2D value: {int(np.round(hr_17,0))}")
print(f"ga2m hour=17pm, year=2012 heatmap value: {int(np.round(hr_17_year_12,0))}")
print(f"ga2m hour=17pm, day=non-working heatmap value: {int(np.round(hr_17_day_no_work,0))}")

### GA2M 1D & Heatmap values at hour = 20:00 

In [ ]:
hr_feat_idx = features.index("hr")
hr_value = 20.00
year_value = 1  # 2012
workingday_value = 0  # non-working day

hr_20 = ga2m_shape_value(
    ga2m, feat_idx=hr_feat_idx, feature_value=hr_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

hr_20_year_12 = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    feature_value=hr_value,
    other_value=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

hr_20_day_no_work = ga2m_interaction_value(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    feature_value=hr_value,
    other_value=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour=20pm 2D value: {int(np.round(hr_20,0))}")
print(f"ga2m hour=20pm, year=2012 heatmap value: {int(np.round(hr_20_year_12,0))}")
print(f"ga2m hour=20pm, day=non-working heatmap value: {int(np.round(hr_20_day_no_work,0))}")

### Q16.b. Suppose it is a non-working day with temperature > 11°C and Year = 2012. What is the approximate change in the prediction when hour increases from 17:00 to 20:00?

In [ ]:
min_hour_value = 17.0
max_hour_value = 20.0

delta_hr_17_20 = ga2m_shift_1d(
    ga2m, feat_idx=hr_feat_idx, k1=min_hour_value, k2=max_hour_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    center_like_plot=True,
    t=t,
)

delta_hr_17_20_year_12 = ga2m_shift_interaction_wrt_feat(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("yr"),
    k1=min_hour_value,
    k2=max_hour_value,
    other_value_fixed=year_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

delta_hr_17_20_day_no_work = ga2m_shift_interaction_wrt_feat(
    ga2m,
    feat_idx=hr_feat_idx,
    other_idx=features.index("workingday"),
    k1=min_hour_value,
    k2=max_hour_value,
    other_value_fixed=workingday_value,
    feat_labels=features,
    categorical_features=categorical_features,
    numerical_features=numerical_features,
    t=t,
)

print(f"ga2m hour shift 17am->20pm 1D delta: {int(np.round(delta_hr_17_20,0))}")
print(f"ga2m hour shift 17am->20pm, year=2012 heatmap delta: {int(np.round(delta_hr_17_20_year_12,0))}")
print(f"ga2m hour shift 17am->20pm, day=non-working heatmap delta: {int(np.round(delta_hr_17_20_day_no_work,0))}")

print(f"Total shift 17am->20pm, year=2012, non-working day: {int(np.round(delta_hr_17_20 + delta_hr_17_20_year_12 + delta_hr_17_20_day_no_work,0))}")

### Q17.b Suppose it is a non-working day with temperature > 11°C and Year = 2012. The current hour is 15:00. What is the shortest amount of time it takes for the predicted rentals to drop by 50?

In [ ]:
year_value = 0  # 2011
min_hour_value = 15
max_hour_value = 22.5
step = 0.1
ga2m_results = []

for end_hour in np.arange(min_hour_value + step, max_hour_value, step):

    delta_hr = ga2m_shift_1d(
        ga2m, feat_idx=hr_feat_idx, k1=min_hour_value, k2=end_hour,
        feat_labels=features,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        center_like_plot=True,
        t=t,
    )

    delta_hr_year_11 = ga2m_shift_interaction_wrt_feat(
        ga2m,
        feat_idx=hr_feat_idx,
        other_idx=features.index("yr"),
        k1=min_hour_value,
        k2=end_hour,
        other_value_fixed=year_value,
        feat_labels=features,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        t=t,
    )

    delta_hr_day_no_work = ga2m_shift_interaction_wrt_feat(
        ga2m,
        feat_idx=hr_feat_idx,
        other_idx=features.index("workingday"),
        k1=min_hour_value,
        k2=end_hour,
        other_value_fixed=workingday_value,
        feat_labels=features,
        categorical_features=categorical_features,
        numerical_features=numerical_features,
        t=t,
    )

    ga2m_results.append({
        "out_shift": delta_hr + delta_hr_year_11 + delta_hr_day_no_work,
        "hr_shift": end_hour - min_hour_value,
    })

In [ ]:
best_result = get_min_hr_shift_for_y_shift(ga2m_results, y_shift=-50)
print(f"total delta: {int(np.round(best_result['out_shift'],0))}, total hr shift: {best_result['hr_shift']*60:.2f} mins")

In [ ]:
plot_hr_shift_y_shift(ga2m_results)